# 05. Backtesting
Chạy backtest ML Full, ML Base, Buy&Hold. Kết quả nhanh + save.

In [1]:
import sys, logging, warnings
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.INFO, format='%(levelname)-8s | %(name)s | %(message)s')
warnings.filterwarnings('ignore')

from src.utils.io import save, load
from src.config import load_config
from src.backtest import BacktestEngineConfig, run_backtest, compute_benchmark, compute_metrics, compute_alpha_stats

cfg = load_config(ROOT / 'configs' / 'base.yaml')
print(f'top_k={cfg.strategy.top_k}, rebalance={cfg.strategy.rebalance_days}d, cost={cfg.backtest.cost_bps}bps')


top_k=10, rebalance=10d, cost=10bps


## 1. Load data

In [2]:
df = load(cfg.dir_processed / 'dataset_labeled.parquet')
pred_ens_full = load(cfg.dir_processed / 'predictions_ens_full.parquet')
pred_ens_base = load(cfg.dir_processed / 'predictions_ens_base.parquet')
print(f'Dataset: {df.shape}, Pred Full: {pred_ens_full.shape}, Pred Base: {pred_ens_base.shape}')


Dataset: (100116, 79), Pred Full: (49303, 5), Pred Base: (49303, 5)


## 2. Run backtest

In [3]:
bt_cfg = BacktestEngineConfig(
    top_k=cfg.strategy.top_k,
    rebalance_days=cfg.strategy.rebalance_days,
    cost_bps=cfg.backtest.cost_bps,
    slippage_bps=cfg.backtest.slippage_bps,
    initial_capital=cfg.backtest.initial_capital,
)

eq_full, trades_full = run_backtest(df, pred_ens_full, bt_cfg)
eq_base, trades_base = run_backtest(df, pred_ens_base, bt_cfg)
eq_bench = compute_benchmark(df, eq_full.index, initial_capital=bt_cfg.initial_capital)

print(f'ML Full : ${eq_full["equity"].iloc[-1]:,.0f}')
print(f'ML Base : ${eq_base["equity"].iloc[-1]:,.0f}')
print(f'Buy&Hold: ${eq_bench["equity"].iloc[-1]:,.0f}')


INFO     | src.backtest.engine | Backtest: 142 rebalances, every 10 days, 2020-01-31→2026-02-26
INFO     | src.backtest.engine |   Final: $45,741 (return: 357.4%)
INFO     | src.backtest.engine | Backtest: 142 rebalances, every 10 days, 2020-01-31→2026-02-26
INFO     | src.backtest.engine |   Final: $38,794 (return: 287.9%)


ML Full : $45,741
ML Base : $38,794
Buy&Hold: $42,678


## 3. Quick metrics

In [4]:
m_full = compute_metrics(eq_full)
m_base = compute_metrics(eq_base)
m_bench = compute_metrics(eq_bench)

metrics_df = pd.DataFrame({'ML_Full': m_full, 'ML_Base': m_base, 'Buy&Hold': m_bench}).T
print(metrics_df[['CAGR', 'Sharpe', 'Sortino', 'Max_Drawdown', 'Calmar']].round(4))


            CAGR  Sharpe  Sortino  Max_Drawdown  Calmar
ML_Full   0.2849  0.9459   1.2927       -0.2415  1.1797
ML_Base   0.2505  0.8845   1.2012       -0.2394  1.0465
Buy&Hold  0.2703  0.8886   1.2201       -0.3546  0.7623


## 4. Save

In [5]:
(cfg.dir_outputs / 'metrics').mkdir(parents=True, exist_ok=True)

metrics_df.to_csv(cfg.dir_outputs / 'metrics' / 'backtest_metrics.csv')

alpha_df = pd.DataFrame({
    'ML_Full': compute_alpha_stats(eq_full, eq_bench),
    'ML_Base': compute_alpha_stats(eq_base, eq_bench),
}).T
alpha_df.to_csv(cfg.dir_outputs / 'metrics' / 'alpha_stats.csv')

save(eq_full, cfg.dir_outputs / 'equity_full.parquet')
save(eq_base, cfg.dir_outputs / 'equity_base.parquet')
save(eq_bench, cfg.dir_outputs / 'equity_benchmark.parquet')
trades_full.to_csv(cfg.dir_outputs / 'metrics' / 'trade_log_full.csv', index=False)
trades_base.to_csv(cfg.dir_outputs / 'metrics' / 'trade_log_base.csv', index=False)

print('✓ All saved → chạy 06_analysis.ipynb để phân tích chi tiết')


✓ All saved → chạy 06_analysis.ipynb để phân tích chi tiết
